In [1]:
!pip install -q "unsloth>=2024.5.3" "transformers>=4.41.0" "datasets>=2.19.0" "accelerate>=0.30.0" "bitsandbytes>=0.43.0" peft


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.7/311.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.8/184.8 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 18.7 MB/s eta 0:00:00


In [2]:
import os, random, torch, math
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Any

from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

from peft import LoraConfig
from unsloth import FastLanguageModel

SEED = 3407
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()
print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


/tmp/ipython-input-4050055275.py:15: UserWarning: WARNING: Unsloth should be imported before transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
CUDA: Tesla T4


In [3]:

model_name = "unsloth/mistral-7b-instruct-v0.2"


MAX_SEQ_LEN = 2048

lora_r = 16
lora_alpha = 32
lora_dropout = 0.1
target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]


In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)


model = FastLanguageModel.get_peft_model(
    model,
    r = lora_r,
    target_modules = target_modules,
    lora_alpha = lora_alpha,
    lora_dropout = lora_dropout,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = SEED,
)


tokenizer.padding_side = "right"
tokenizer.truncation_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Model & tokenizer ready.")


==((====))==  Unsloth 2025.8.9: Fast Mistral patching. Transformers: 4.55.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.8.9 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Model & tokenizer ready.


In [5]:
import json
from datasets import Dataset

with open("/content/ipc_qa.json") as f:
    data = json.load(f)

raw_ds = Dataset.from_list(data)
print(raw_ds, raw_ds[0])


Dataset({
    features: ['question', 'answer'],
    num_rows: 2267
}) {'question': 'What is the title and extent of operation of the Indian Penal Code?', 'answer': "The title is 'The Indian Penal Code' and its operation extends to the punishment of offences committed within India, and beyond but which by law may be tried within India. It also includes extension of the Code to extra-territorial offences."}


In [7]:
ds = DatasetDict({"train": raw_ds})

In [8]:

def build_prompt(question, context="", answer=None):
    if context and isinstance(context, str) and context.strip():
        prompt = (
            "You are a legal assistant. Provide accurate, concise, and well-cited answers when possible.\n\n"
            f"Question: {question}\n"
            f"Context: {context}\n"
            "Answer:"
        )
    else:
        prompt = (
            "You are a legal assistant. Provide accurate, concise, and jurisdiction-aware answers.\n\n"
            f"Question: {question}\n"
            "Answer:"
        )
    if answer is not None:
        return f"{prompt} {answer}".strip()
    return prompt.strip()

def to_text(example):
    q = example.get("question","").strip()
    a = example.get("answer","").strip()
    c = example.get("context","")
    return {"text": build_prompt(q, c, a)}

formatted = ds["train"].map(to_text, remove_columns=ds["train"].column_names)
formatted = formatted.shuffle(seed=SEED)
print(formatted[0])


Map:   0%|          | 0/2267 [00:00<?, ? examples/s]

{'text': 'You are a legal assistant. Provide accurate, concise, and jurisdiction-aware answers.\n\nQuestion: Which section of law mentions the offense of grievous hurt?\nAnswer: Section 320'}


In [9]:
split = formatted.train_test_split(test_size=0.05, seed=SEED)
train_ds, val_ds = split["train"], split["test"]

def tok(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
        return_attention_mask=True,
    )

train_tok = train_ds.map(tok, batched=True, remove_columns=["text"])
val_tok   = val_ds.map(tok, batched=True, remove_columns=["text"])

print(train_tok[0].keys())
print(f"Train size: {len(train_tok)}, Val size: {len(val_tok)}")


Map:   0%|          | 0/2153 [00:00<?, ? examples/s]

Map:   0%|          | 0/114 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask'])
Train size: 2153, Val size: 114


In [10]:

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


In [12]:
!pip install -q -U transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 119.4 MB/s eta 0:00:00


In [13]:
EPOCHS = 2
BSZ = 2
GRAD_ACCUM = 8
LR = 2e-4
MAX_STEPS = -1
SAVE_STEPS = 200
LOG_STEPS = 20
WARMUP_STEPS = 50

args = TrainingArguments(
    output_dir = "/content/legal-llm",
    num_train_epochs = EPOCHS,
    per_device_train_batch_size = BSZ,
    per_device_eval_batch_size = BSZ,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate = LR,
    warmup_steps = WARMUP_STEPS,
    lr_scheduler_type = "cosine",
    logging_steps = LOG_STEPS,
    eval_strategy = "steps",
    eval_steps = SAVE_STEPS,
    save_steps = SAVE_STEPS,
    save_total_limit = 2,
    bf16 = torch.cuda.is_bf16_supported(),
    fp16 = not torch.cuda.is_bf16_supported(),
    dataloader_num_workers = 2,
    optim = "adamw_8bit",
    max_steps = MAX_STEPS,
    report_to = "none",
    gradient_checkpointing = True,
    logging_first_step = True,
)
print(args)


TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=2,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=200,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=False,

In [14]:
def compute_metrics(eval_pred):

    return {}

trainer = Trainer(
    model = model,
    args = args,
    train_dataset = train_tok,
    eval_dataset = val_tok,
    data_collator = collator,
)

train_result = trainer.train()
trainer.save_state()
trainer.save_model("/content/legal-llm/checkpoint-last")
print("Training finished.")


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,153 | Num Epochs = 2 | Total steps = 270
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
200,0.415100,0.544820


Unsloth: Not an error, but MistralForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Training finished.


In [16]:
from torch.utils.data import DataLoader

val_ds = formatted.select(range(min(20, len(formatted))))

sample_loader = DataLoader(val_ds.select(range(min(3, len(val_ds)))), batch_size=1)

for i, ex in enumerate(sample_loader):
    prompt = ex["text"][0]
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        top_p=0.95,
        temperature=0.7
    )
    print(f"\n--- Example {i+1} ---")
    print(tokenizer.decode(out[0], skip_special_tokens=True))



--- Example 1 ---
You are a legal assistant. Provide accurate, concise, and jurisdiction-aware answers.

Question: Which section of law mentions the offense of grievous hurt?
Answer: Section 320 mentions the offense of grievous hurt. This includes causing hurt by means of poison, causing hurt by means of any instrument or substance which, when used as a weapon of offence, causes death, and causing hurt by means of any instrument or substance which, when used as a weapon of offence, causes death. It also includes attempting to cause any hurt as described above. However, this does not include any hurt which is not grievous. This offense can be punished with imprisonment of either description for a term which may extend to seven years, or with imprisonment of either description for a term which may extend to three years, or with fine, or with both. In the event of death, the offender can also be punished with death or imprisonment for life. If the offender is a public servant, he can be 

In [17]:
ADAPTER_DIR = "/content/legal-llm/lora-adapter"
MERGED_DIR  = "/content/legal-llm/merged"


trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved LoRA adapters to:", ADAPTER_DIR)

try:
    merged = trainer.model.merge_and_unload()
    merged.save_pretrained(MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_DIR)
    print("Saved MERGED model to:", MERGED_DIR)
except Exception as e:
    print("Merge failed (likely VRAM or base model restrictions). Use adapters:", e)


Saved LoRA adapters to: /content/legal-llm/lora-adapter


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:348: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Saved MERGED model to: /content/legal-llm/merged


In [18]:

base, tok = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)
from peft import PeftModel
inf_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
FastLanguageModel.for_inference(inf_model)

def answer(question, context=""):
    prompt = build_prompt(question, context, answer=None)
    inputs = tok(prompt, return_tensors="pt").to("cuda")
    out = inf_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        eos_token_id=tok.eos_token_id,
    )
    return tok.decode(out[0], skip_special_tokens=True)

print(answer("What is res judicata under CPC?"))


==((====))==  Unsloth 2025.8.9: Fast Mistral patching. Transformers: 4.55.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
You are a legal assistant. Provide accurate, concise, and jurisdiction-aware answers.

Question: What is res judicata under CPC?
Answer: Res judicata under CPC refers to a matter that has been judicially determined, and is therefore not subject to further legal action. This principle is intended to prevent the same matter from being tried again. It also prevents a matter from being tried on the same facts and issues again. However, this does not extend to a question of law. It is a question of fact whether a particular matter has been j

In [20]:

print(answer("What is 'Man', 'Woman', 'Person', and 'Public' according to the Indian Penal Code?"))

You are a legal assistant. Provide accurate, concise, and jurisdiction-aware answers.

Question: What is 'Man', 'Woman', 'Person', and 'Public' according to the Indian Penal Code?
Answer: These terms are defined in the 'General Explanations' section of the Indian Penal Code. However, the actual definitions are not provided in the provided text. It is recommended to consult the actual text of the Indian Penal Code for the definitions of these terms.
